# OCR stiker obat - end to end v2 (YOLO 2 kelas + PaddleOCR)
Alur: **YOLO 2 kelas (ambil `entete`) → orientasi Paddle (0/90/180/270) → putar + enhance →
PaddleOCR → teks**. Notebook patokan soal vignette (penyisihan GEMASTIK XIX).

- **YOLO**: 2 kelas (`entete`, `vin`), augmentasi rotasi (`degrees=180`, `flipud=0.5`),
  `CONF = 0.1`, hanya kotak `entete` yang dipakai.
- **Label** ditulis ke `labels_v2.csv`; GUI bisa menampilkan **draf PaddleOCR** untuk diperiksa.
- **Orientasi**: classifier bawaan Paddle + kalibrasi (tidak dilatih).

| bagian | isi | catatan |
|---|---|---|
| 0 | konfigurasi & fungsi bantu | selalu jalan |
| 1 | siapkan data (YOLO 2 kelas + crop + `labels_v2.csv`) | sekali saja |
| 2 | labeling manual (draf Paddle opsional) | `rot` untuk kalibrasi, `text` untuk CER |
| 3 | latih YOLO | dilewati kalau `yolo_2cls.pt` sudah ada |
| 4 | kalibrasi orientasi | memetakan label Paddle ke rotasi koreksi |
| 5 | ukur PaddleOCR (CER) | memilih `ENHANCE` |
| 6 | inference | `submission_v2.csv` + `debug_v2.csv` |

Semua hasil (dataset YOLO, crop, label, model, submission) ditulis ke `WORK_FOLDER`.
Paket: `ultralytics rapidfuzz paddlepaddle paddleocr tqdm`. PaddleOCR mengunduh modelnya sekali
saat online (cache `~/.paddlex/official_models`); salin cache itu sebelum lomba.

## 0. Konfigurasi & fungsi bantu

In [ ]:
import os
import random

import numpy as np
from tqdm.auto import tqdm

DATA_FOLDER = "../../../data"   # UBAH: berisi train/ (COCO), test/, sample_submission.csv
PRETRAINED_YOLO = "../../../pretrained/yolov8n/yolov8n.pt"   # UBAH
WORK_FOLDER = os.path.abspath("../../../outputs/pipeline/ocr_end_to_end_v2")   # UBAH
WORK_FOLDER = WORK_FOLDER.replace("\\", "/")
CROPS_FOLDER = f"{WORK_FOLDER}/crops"
TEST_CROPS_FOLDER = f"{WORK_FOLDER}/test_crops_v2"
YOLO_FOLDER = f"{WORK_FOLDER}/yolo2"
MODEL_FOLDER = f"{WORK_FOLDER}/models"
LABEL_CSV = f"{WORK_FOLDER}/labels_v2.csv"
for folder in [CROPS_FOLDER, TEST_CROPS_FOLDER, MODEL_FOLDER]:
    os.makedirs(folder, exist_ok=True)

SEED = 42
CLASS_NAMES = ["entete", "vin"]   # urutan = nomor kelas YOLO (entete = 0, vin = 1)
COCO_TO_YOLO = {1: 0, 2: 1}       # COCO: 0 = 'Images' (dummy), 1 = 'entete', 2 = 'vin'
PAD = 4                           # UBAH: padding crop (px)
IMGSZ = 416                       # UBAH: ukuran input YOLO
CONF = 0.1                        # UBAH: ambang confidence YOLO
UPSCALE = 3                       # UBAH: crop diperbesar dulu karena tulisannya kecil
random.seed(SEED)
np.random.seed(SEED)
print("hasil ditulis ke", WORK_FOLDER)

In [ ]:
import re

import cv2

SKIP = "__SKIP__"   # penanda crop yang dilewati saat labeling
# KONVENSI: rot = derajat putar SEARAH JARUM JAM agar teks tegak
CV_ROTATE = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180,
             270: cv2.ROTATE_90_COUNTERCLOCKWISE}


def clean_text(text):
    # huruf besar, buang semua selain A-Z 0-9 / - (spasi, ':', '.', dll ikut dibuang)
    return re.sub(r"[^A-Z0-9/\-]", "", str(text).upper())


def rotate_cw(image, rot):
    if rot % 360 == 0:
        return image
    return cv2.rotate(image, CV_ROTATE[rot % 360])


def needed_rot(applied):
    # gambar tegak yang diputar `applied` derajat cw butuh berapa derajat cw untuk tegak lagi
    return (360 - applied) % 360


print(clean_text("Lot: gi005 fab.10/19"), needed_rot(90))   # → LOTGI005FAB10/19 270

In [ ]:
ENHANCE = True   # CLAHE + sharpen tipis sebelum OCR, dipilih ulang di bagian 5


def enhance(image, use_enhance=None):
    # perbaiki kontras (CLAHE di kanal L) + sharpen tipis; None → ikut ENHANCE
    if use_enhance is None:
        use_enhance = ENHANCE
    if not use_enhance:
        return image
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    lightness, green_red, blue_yellow = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    lightness = clahe.apply(lightness)
    image = cv2.cvtColor(cv2.merge([lightness, green_red, blue_yellow]), cv2.COLOR_LAB2BGR)
    blurred = cv2.GaussianBlur(image, (0, 0), 1.0)
    return cv2.addWeighted(image, 1.4, blurred, -0.4, 0)


print(enhance(np.zeros((8, 8, 3), dtype=np.uint8)).shape)

In [ ]:
def center_y(item):
    return item["cy"]


def left_x(item):
    return item["x"]


def ocr_items(results):
    # results = [(poligon 4 titik, teks), ...] → pusat-y, tinggi, kiri-x, teks per kotak
    items = []
    for box, text in results:
        xs = []
        ys = []
        for point in box:
            xs.append(point[0])
            ys.append(point[1])
        items.append({"cy": sum(ys) / 4, "h": max(ys) - min(ys), "x": min(xs), "text": text})
    return items


demo_results = [([[50, 0], [90, 0], [90, 10], [50, 10]], "B"),
                ([[0, 1], [40, 1], [40, 11], [0, 11]], "A"),
                ([[0, 30], [40, 30], [40, 40], [0, 40]], "C")]
print(ocr_items(demo_results)[0])

In [ ]:
def merge_lines(results):
    # urutkan: baris atas → bawah, dalam satu baris kiri → kanan; hasil = teks digabung
    items = ocr_items(results)
    items.sort(key=center_y)
    lines = []
    for item in items:
        if lines and abs(item["cy"] - lines[-1][0]["cy"]) < 0.6 * max(item["h"], 1):
            lines[-1].append(item)   # masih satu baris
        else:
            lines.append([item])
    text = ""
    for line in lines:
        line.sort(key=left_x)
        for item in line:
            text = text + item["text"]
    return text


print(merge_lines(demo_results))   # → ABC

In [ ]:
PADDLE = {}   # model PaddleOCR dimuat sekali saja


def get_paddle():
    # PaddleOCR (deteksi + recognisi teks); butuh internet sekali untuk mengunduh modelnya
    if "ocr" not in PADDLE:
        from paddleocr import PaddleOCR
        PADDLE["ocr"] = PaddleOCR(lang="en", use_doc_orientation_classify=False,
                                  use_doc_unwarping=False, use_textline_orientation=False,
                                  enable_mkldnn=False)
    return PADDLE["ocr"]


def get_ori():
    # classifier orientasi dokumen bawaan PaddleOCR (0/90/180/270)
    if "ori" not in PADDLE:
        from paddleocr import DocImgOrientationClassification
        PADDLE["ori"] = DocImgOrientationClassification(model_name="PP-LCNet_x1_0_doc_ori")
    return PADDLE["ori"]


def paddle_field(result, key):
    # versi PaddleOCR berbeda membungkus hasil sedikit berbeda
    if isinstance(result, dict) and key in result:
        return result[key]
    return result.json["res"][key]


print(paddle_field({"rec_texts": ["LOT"]}, "rec_texts"))

In [ ]:
ORI_MAP = {0: 0, 90: 90, 180: 180, 270: 270}   # label Paddle → derajat cw koreksi (bagian 4)


def paddle_read(upright, use_enhance=None):
    # baca crop yang SUDAH tegak → teks bersih (A-Z 0-9 / -)
    image = enhance(upright, use_enhance)
    big = cv2.resize(image, None, fx=UPSCALE, fy=UPSCALE, interpolation=cv2.INTER_CUBIC)
    results = []
    for result in get_paddle().predict(big):
        polys = paddle_field(result, "rec_polys")
        texts = paddle_field(result, "rec_texts")
        for poly, text in zip(polys, texts):
            results.append((poly, text))
    return clean_text(merge_lines(results))


def ori_label(image):
    # label mentah orientasi dari Paddle (0/90/180/270); arti arahnya dikalibrasi di bagian 4
    result = next(iter(get_ori().predict(image, batch_size=1)))
    return int(paddle_field(result, "label_names")[0])


print("fungsi Paddle siap (model dimuat saat pertama dipakai)")

In [ ]:
import pandas as pd


def load_labeled():
    # baris labels_v2.csv yang sudah terisi (tanpa yang dilewati)
    table = pd.read_csv(LABEL_CSV, dtype=str, keep_default_na=False)
    table = table[(table["text"] != "") & (table["text"] != SKIP)].copy()
    table["rot"] = table["rot"].astype(int)
    return table


def load_upright(file_name, rot):
    # crop yang sudah diputar sehingga teks tegak
    return rotate_cw(cv2.imread(f"{CROPS_FOLDER}/{file_name}"), rot)


def cer(preds, refs):
    # Character Error Rate = total jarak Levenshtein / total panjang referensi
    from rapidfuzz.distance import Levenshtein
    errors = 0
    length = 0
    for pred, ref in zip(preds, refs):
        errors = errors + Levenshtein.distance(pred, ref)
        length = length + len(ref)
    return errors / max(1, length)


print("CER contoh:", cer(["LOT1"], ["LOT7"]))   # → 0.25

In [ ]:
def best_entete(classes, confs):
    # indeks kotak kelas entete (0) dengan conf tertinggi (None kalau tidak ada) + jumlahnya
    best = None
    count = 0
    for index in range(len(classes)):
        if int(classes[index]) == 0:
            count = count + 1
            if best is None or confs[index] > confs[best]:
                best = index
    return best, count


print(best_entete([1, 0, 0], [0.9, 0.3, 0.6]))   # → (2, 2)

## 1. Siapkan data
Membuat (a) dataset YOLO **dua kelas** (`entete` = 0, `vin` = 1) di `yolo2/`, (b) crop `entete`
dari kotak train + `labels_v2.csv` kosong untuk diisi.
**Aman dijalankan ulang:** `labels_v2.csv` tidak ditimpa kalau sudah ada (label tidak hilang).

In [ ]:
import json

with open(f"{DATA_FOLDER}/train/_annotations.coco.json", encoding="utf-8") as file:
    coco = json.load(file)
for annotation in coco["annotations"]:
    if "image_id" not in annotation:   # perbaiki 1 anotasi rusak (key '' dan salah ketik)
        annotation["image_id"] = annotation.pop("")
        annotation["category_id"] = annotation.pop("catimage_idegory_id")

images = {}
for image in coco["images"]:
    images[image["id"]] = image
anns = []          # tanpa kelas dummy 'Images'
entete_anns = []
anns_by_image = {}
for annotation in coco["annotations"]:
    if annotation["category_id"] in COCO_TO_YOLO:
        anns.append(annotation)
        anns_by_image.setdefault(annotation["image_id"], []).append(annotation)
        if COCO_TO_YOLO[annotation["category_id"]] == 0:
            entete_anns.append(annotation)
print("gambar:", len(images), "| kotak entete:", len(entete_anns),
      "| kotak vin:", len(anns) - len(entete_anns))

In [ ]:
def yolo_line(annotation, width, height):
    # kotak COCO (kiri, atas, lebar, tinggi) piksel → "kelas x_tengah y_tengah lebar tinggi" (0–1)
    left, top, box_width, box_height = annotation["bbox"]
    number = COCO_TO_YOLO[annotation["category_id"]]
    x_center = (left + box_width / 2) / width
    y_center = (top + box_height / 2) / height
    box_width = box_width / width
    box_height = box_height / height
    return f"{number} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}"


print(yolo_line({"bbox": [10, 20, 30, 40], "category_id": 1}, 100, 200))   # → 0 0.25 0.2 0.3 0.2

In [ ]:
import shutil

image_ids = sorted(images)
random.Random(SEED).shuffle(image_ids)
val_ids = set(image_ids[: int(0.2 * len(image_ids))])   # bagi per GAMBAR supaya tidak bocor

shutil.rmtree(YOLO_FOLDER, ignore_errors=True)   # dibuat ulang dari nol setiap dijalankan
for part in ["train", "val"]:
    os.makedirs(f"{YOLO_FOLDER}/images/{part}")
    os.makedirs(f"{YOLO_FOLDER}/labels/{part}")
for image_id, image in images.items():
    split = "train"
    if image_id in val_ids:
        split = "val"
    shutil.copy(f"{DATA_FOLDER}/train/{image['file_name']}", f"{YOLO_FOLDER}/images/{split}")
    lines = []
    for annotation in anns_by_image.get(image_id, []):
        lines.append(yolo_line(annotation, image["width"], image["height"]))
    stem = os.path.splitext(image["file_name"])[0]
    with open(f"{YOLO_FOLDER}/labels/{split}/{stem}.txt", "w") as file:
        file.write("\n".join(lines))
with open(f"{YOLO_FOLDER}/data.yaml", "w") as file:
    file.write(f"path: {YOLO_FOLDER}\ntrain: images/train\nval: images/val\n"
               "names:\n  0: entete\n  1: vin\n")
print("train:", len(os.listdir(f"{YOLO_FOLDER}/images/train")),
      "| val:", len(os.listdir(f"{YOLO_FOLDER}/images/val")))

In [ ]:
if os.path.exists(LABEL_CSV):
    print("labels_v2.csv sudah ada → crop & label TIDAK ditimpa.")
else:
    rows = []
    for annotation in entete_anns:
        image = images[annotation["image_id"]]
        picture = cv2.imread(f"{DATA_FOLDER}/train/{image['file_name']}")
        left, top, box_width, box_height = annotation["bbox"]
        left, top = int(left), int(top)
        right, bottom = left + int(box_width), top + int(box_height)
        crop = picture[max(0, top - PAD): bottom + PAD, max(0, left - PAD): right + PAD]
        stem = os.path.splitext(image["file_name"])[0]
        name = f"{stem}_{annotation['id']}.jpg"
        cv2.imwrite(f"{CROPS_FOLDER}/{name}", crop)
        split = "train"
        if annotation["image_id"] in val_ids:
            split = "val"
        rows.append({"file": name, "split": split, "rot": "", "text": ""})
    pd.DataFrame(rows).to_csv(LABEL_CSV, index=False)
    print("crop tersimpan:", len(rows), "→", LABEL_CSV)

## 2. Labeling manual
**Format label (asumsi, belum dikonfirmasi panitia):** teks penuh persis yang tercetak di entete,
atas ke bawah, termasuk kata kunci, tanpa spasi. Contoh: `LOTGI005FAB10/19PER10/21`. Bentuk ini
sama dengan keluaran Paddle, jadi label langsung sebanding dengan hasil OCR.

**2a. Draf otomatis (opsional, lambat).** Untuk `DRAFT_N` crop pertama (urutan sama dengan GUI),
PaddleOCR menebak rotasi (90 atau 270, karena entete selalu vertikal) dan teksnya. Hasil di
`drafts.csv`, bisa dilanjutkan kalau terhenti. `DRAFT_N = 0` untuk mengetik dari nol.

In [ ]:
DRAFT_CSV = f"{WORK_FOLDER}/drafts.csv"
DRAFT_N = 50   # UBAH: berapa crop pertama yang dibuatkan draf (±5–8 detik per crop di CPU)
KEYWORDS = r"LOT|FAB|EXP|PER|DDF|DDP|DE"


def label_order(table):
    # urutan crop di GUI (diacak supaya train & val sama-sama terisi); dipakai juga oleh draf
    order = list(table.index)
    random.Random(SEED).shuffle(order)
    return order


def draft_for(file_name):
    # coba rotasi 90 dan 270, pilih bacaan paling mirip stiker (banyak kata kunci, lalu terpanjang)
    image = cv2.imread(f"{CROPS_FOLDER}/{file_name}")
    best = (-1, 0, "")
    for rot in [90, 270]:
        text = paddle_read(rotate_cw(image, rot))
        score = 100 * len(re.findall(KEYWORDS, text)) + len(text)
        if score > best[0]:
            best = (score, rot, text)
    return best[1], best[2]


print(label_order(pd.DataFrame({"file": ["a", "b", "c"]})))

In [ ]:
def read_drafts():
    # draf yang sudah ada (bisa dilanjutkan kalau terhenti)
    if os.path.exists(DRAFT_CSV):
        return pd.read_csv(DRAFT_CSV, dtype=str, keep_default_na=False).to_dict("records")
    return []


def make_drafts(count):
    table = pd.read_csv(LABEL_CSV, dtype=str, keep_default_na=False)
    rows = read_drafts()
    have = set()
    for row in rows:
        have.add(row["file"])
    todo = []
    for index in label_order(table)[:count]:
        if table.at[index, "text"] == "" and table.at[index, "file"] not in have:
            todo.append(table.at[index, "file"])
    for file_name in tqdm(todo, desc="draf Paddle"):
        rot, text = draft_for(file_name)
        rows.append({"file": file_name, "rot": str(rot), "text": text})
        if len(rows) % 10 == 0:
            pd.DataFrame(rows).to_csv(DRAFT_CSV, index=False)
    pd.DataFrame(rows).to_csv(DRAFT_CSV, index=False)
    print("draf tersedia:", len(rows))


if DRAFT_N > 0:
    make_drafts(DRAFT_N)

**2b. GUI labeling** (satu jendela Tkinter). Per crop:
1. Kalau ada draf, tampil teks tebakan Paddle dan gambar sudah diputar sesuai tebakannya.
   **Periksa dua hal**: gambar benar-benar tegak (belum: **Ctrl+R** putar 90° searah jarum jam),
   dan teksnya benar huruf per huruf. Draf sering salah di `0/O`, `1/I`, dan urutan baris.
2. **Enter** menyimpan (`rot` dan `text`). 3. Tidak terbaca sama sekali: **Lewati**.

Tersimpan tiap Enter; boleh ditutup dan dibuka lagi. Target awal **≥ 100 crop** (minimal 20
untuk kalibrasi orientasi). Jangan Enter beruntun tanpa memeriksa: label yang diterima buta
membuat CER dan kalibrasi terlihat lebih bagus dari kenyataannya.

In [ ]:
from PIL import Image, ImageTk

VIEW_W, VIEW_H = 900, 420
gui = {}   # jendela, widget, dan keadaan GUI labeling


def gui_load():
    gui["table"] = pd.read_csv(LABEL_CSV, dtype=str, keep_default_na=False)
    gui["order"] = label_order(gui["table"])
    gui["drafts"] = {}
    for row in read_drafts():
        gui["drafts"][row["file"]] = (int(row["rot"]), row["text"])
    gui["pos"] = 0
    for position, index in enumerate(gui["order"]):
        if gui["table"].at[index, "text"] == "":   # mulai dari crop pertama yang belum dilabel
            gui["pos"] = position
            break


def gui_show():
    row = gui["table"].loc[gui["order"][gui["pos"]]]
    picture = Image.open(f"{CROPS_FOLDER}/{row['file']}").convert("RGB")
    picture = picture.rotate(-gui["rot"], expand=True)   # -rot = searah jarum jam
    scale = min(VIEW_W / picture.width, VIEW_H / picture.height)
    size = (int(picture.width * scale), int(picture.height * scale))
    gui["photo"] = ImageTk.PhotoImage(picture.resize(size, Image.LANCZOS))
    gui["canvas"].config(image=gui["photo"])
    done = int((gui["table"]["text"] != "").sum())
    gui["info"].config(text=f"[{gui['pos'] + 1}/{len(gui['order'])}]  {row['file']}  |  "
                            f"rot = {gui['rot']}  |  terlabel: {done}")

In [ ]:
def saved_label(row):
    # rotasi & teks yang sudah tersimpan untuk satu baris label
    rot = 0
    if row["rot"] != "":
        rot = int(row["rot"])
    text = row["text"]
    if text == SKIP:
        text = ""
    return rot, text


def gui_goto(position):
    gui["pos"] = max(0, min(len(gui["order"]) - 1, position))
    row = gui["table"].loc[gui["order"][gui["pos"]]]
    note = ""
    if row["text"] == "" and row["file"] in gui["drafts"]:   # belum dilabel, ada draf Paddle
        gui["rot"], text = gui["drafts"][row["file"]]
        note = "DRAF PaddleOCR: cek rotasi dan teks huruf per huruf, lalu Enter"
    else:
        gui["rot"], text = saved_label(row)
    gui["text"].set(text)
    gui["note"].config(text=note)
    gui_show()
    gui["entry"].focus_set()


print(saved_label({"rot": "90", "text": SKIP}))   # → (90, '')

In [ ]:
from tkinter import messagebox


def gui_rotate(event=None):
    gui["rot"] = (gui["rot"] + 90) % 360
    gui_show()


def gui_store(rot, text):
    index = gui["order"][gui["pos"]]
    gui["table"].at[index, "rot"] = rot
    gui["table"].at[index, "text"] = text
    gui["table"].to_csv(LABEL_CSV, index=False)
    gui_goto(gui["pos"] + 1)


def gui_save(event=None):
    cleaned = clean_text(gui["text"].get())
    if cleaned == "":
        messagebox.showwarning("Teks kosong", "Isi teksnya dulu, atau pakai Lewati.")
        return
    gui_store(str(gui["rot"]), cleaned)


def gui_skip():
    gui_store("0", SKIP)

In [ ]:
import tkinter as tk


def gui_back():
    gui_goto(gui["pos"] - 1)


def gui_preview(*args):
    gui["preview"].config(text="disimpan sebagai: " + clean_text(gui["text"].get()))


def gui_build():
    root = tk.Tk()
    root.title("Label entete")
    gui["canvas"] = tk.Label(root, bg="#222")
    gui["info"] = tk.Label(root, font=("Arial", 11))
    gui["note"] = tk.Label(root, font=("Arial", 11, "bold"), fg="#c60")
    gui["preview"] = tk.Label(root, font=("Consolas", 12), fg="#0a6")
    gui["text"] = tk.StringVar()
    gui["entry"] = tk.Entry(root, textvariable=gui["text"], font=("Consolas", 16), width=60)
    for name in ["canvas", "info", "note", "preview", "entry"]:
        gui[name].pack(padx=10, pady=4)
    gui["text"].trace_add("write", gui_preview)
    bar = tk.Frame(root)
    bar.pack(pady=6)
    buttons = [("Kembali", gui_back), ("Putar 90 (Ctrl+R)", gui_rotate), ("Lewati", gui_skip),
               ("Simpan & Lanjut (Enter)", gui_save)]
    for label, command in buttons:
        tk.Button(bar, text=label, command=command, width=22).pack(side="left", padx=4)
    return root

In [ ]:
RUN_LABEL_TOOL = True   # UBAH: False kalau tidak sedang melabeli (GUI menahan Run All)
if RUN_LABEL_TOOL:
    gui_load()
    window = gui_build()
    window.bind("<Return>", gui_save)
    window.bind("<Control-r>", gui_rotate)
    gui_goto(gui["pos"])
    window.mainloop()

In [ ]:
labeled = load_labeled()
print("terlabel:", len(labeled), "| train:", (labeled["split"] == "train").sum(),
      "| val:", (labeled["split"] == "val").sum())
print("rot:", labeled["rot"].value_counts().to_dict())
labeled[["file", "rot", "text"]].head(10)

## 3. Latih YOLO (2 kelas, augmentasi rotasi)
`yolov8n`, 30 epoch, `degrees=180`, `flipud=0.5`: train selalu tegak sedangkan test banyak yang
miring/terbalik. `hsv_v` dan `hsv_s` karena test lebih gelap dan tajam (EDA).
mAP val dihitung dari gambar train yang tegak, jadi terlalu optimistis untuk test.
Dilewati otomatis kalau `yolo_2cls.pt` sudah ada.

In [ ]:
from ultralytics import YOLO

RETRAIN_YOLO = False   # UBAH: True = latih ulang (MENIMPA yolo_2cls.pt)
YOLO_WEIGHTS = f"{MODEL_FOLDER}/yolo_2cls.pt"
RUNS_FOLDER = f"{WORK_FOLDER}/runs"

if os.path.exists(YOLO_WEIGHTS) and not RETRAIN_YOLO:
    print("yolo_2cls.pt sudah ada → training dilewati (RETRAIN_YOLO = True untuk melatih ulang)")
else:
    trainer_model = YOLO(PRETRAINED_YOLO)
    trainer_model.train(data=f"{YOLO_FOLDER}/data.yaml", imgsz=IMGSZ, epochs=30,
                        batch=16,   # UBAH: turunkan kalau GPU panas/penuh
                        workers=0, project=RUNS_FOLDER, name="entete2", exist_ok=True,
                        plots=False, seed=SEED,
                        degrees=180, flipud=0.5,   # test banyak yang miring/terbalik
                        hsv_v=0.5, hsv_s=0.5)      # test lebih gelap & tajam (dari EDA)
    shutil.copy(trainer_model.trainer.best, YOLO_WEIGHTS)
    checker = YOLO(YOLO_WEIGHTS)
    metrics = checker.val(data=f"{YOLO_FOLDER}/data.yaml", imgsz=IMGSZ, workers=0, plots=False,
                          project=RUNS_FOLDER, name="val", exist_ok=True)
    print("mAP50:", round(metrics.box.map50, 4), "| mAP50-95:", round(metrics.box.map, 4))
    for number, name in enumerate(CLASS_NAMES):
        print(name, "mAP50:", round(metrics.box.ap50[number], 4))

In [ ]:
# Cek detektor di TEST (tidak punya label): berapa gambar punya kotak entete & seberapa yakin
yolo = YOLO(YOLO_WEIGHTS)
test_paths = []
for file_name in sorted(os.listdir(f"{DATA_FOLDER}/test")):
    if os.path.splitext(file_name)[1].lower() in [".jpg", ".jpeg", ".png"]:
        test_paths.append(f"{DATA_FOLDER}/test/{file_name}")
confs = []   # conf entete tertinggi per gambar
for result in yolo.predict(test_paths, imgsz=IMGSZ, conf=CONF, stream=True, verbose=False):
    best, count = best_entete(result.boxes.cls.tolist(), result.boxes.conf.tolist())
    if best is not None:
        confs.append(float(result.boxes.conf[best]))
share = len(confs) / len(test_paths)
print(f"test: {len(test_paths)} gambar | ada entete: {len(confs)} ({share:.1%})")
print(f"conf median: {np.median(confs):.3f} | min: {min(confs):.3f}")

## 4. Orientasi (PaddleOCR, tanpa training)
Tiap crop berlabel (sudah ditegakkan) diputar 4 arah, lalu classifier Paddle menebak orientasinya.
Arah label Paddle bisa berbeda dengan konvensi `rot` di sini, jadi hasilnya dikalibrasi dengan
voting menjadi `ORI_MAP` (label Paddle → derajat cw koreksi) dan disimpan ke
`models/orientation_v2.json`. Butuh minimal 20 crop berlabel; makin banyak makin bisa dipercaya.

In [ ]:
from collections import Counter, defaultdict

labeled = load_labeled()
if len(labeled) < 20:
    raise ValueError(f"Baru {len(labeled)} crop berlabel - lengkapi dulu di bagian 2.")
votes = defaultdict(Counter)   # votes[label_paddle][rotasi_koreksi_yang_benar]
pairs = list(zip(labeled["file"], labeled["rot"]))
for file_name, rot in tqdm(pairs, desc="kalibrasi orientasi"):
    upright = load_upright(file_name, rot)
    for applied in [0, 90, 180, 270]:
        votes[ori_label(rotate_cw(upright, applied))][needed_rot(applied)] += 1

ORI_MAP = {}
total = 0
correct = 0
for label, counter in votes.items():
    ORI_MAP[label] = counter.most_common(1)[0][0]
    total = total + sum(counter.values())
    correct = correct + counter[ORI_MAP[label]]
for label in sorted(votes):
    print(f"label Paddle {label:3d} → putar cw {ORI_MAP[label]:3d} | sebaran: {dict(votes[label])}")
print(f"akurasi orientasi (setelah kalibrasi): {correct / total:.3f} dari {total} sampel")

ORI_JSON = f"{MODEL_FOLDER}/orientation_v2.json"
ori_saved = {}
for label in ORI_MAP:
    ori_saved[str(label)] = int(ORI_MAP[label])
with open(ORI_JSON, "w") as file:
    json.dump(ori_saved, file)
print("tersimpan →", ORI_JSON)

## 5. Ukur PaddleOCR (CER)
PaddleOCR dipakai **apa adanya** (tidak dilatih), jadi semua crop berlabel boleh jadi alat ukur.
Crop dibaca dengan `rot` dari label, jadi angka ini hanya mengukur kualitas OCR (bukan kegagalan
YOLO atau orientasi di test). Sel ini juga memilih `ENHANCE` yang CER-nya lebih kecil.

In [ ]:
labeled = load_labeled()
if len(labeled) < 20:
    raise ValueError(f"Baru {len(labeled)} crop berlabel - lengkapi dulu di bagian 2.")
refs = list(labeled["text"])
preds_by = {}
for flag in [True, False]:
    preds_by[flag] = []
    for file_name, rot in zip(labeled["file"], labeled["rot"]):
        preds_by[flag].append(paddle_read(load_upright(file_name, rot), use_enhance=flag))
    score = cer(preds_by[flag], refs)
    print(f"PaddleOCR (enhance={flag!s:5}) CER: {score:.4f}   ({len(labeled)} crop)")

ENHANCE = cer(preds_by[True], refs) <= cer(preds_by[False], refs)
print("dipakai di inference: ENHANCE =", ENHANCE)
for pred, ref in zip(preds_by[ENHANCE][:10], refs[:10]):   # lihat dengan mata, bukan cuma CER
    mark = "XX "
    if pred == ref:
        mark = "OK "
    print(mark, "benar:", ref, "\n    tebak:", pred)

## 6. Inference → `submission_v2.csv` dan `debug_v2.csv`
Mandiri setelah bagian 0: butuh `yolo_2cls.pt` dan `orientation_v2.json` di `MODEL_FOLDER`. Dari
tiap gambar test diambil **satu kotak `entete` dengan conf tertinggi**.

- `LIMIT = 200` untuk cek cepat (200 baris pertama `sample_submission.csv`); `None` untuk semua.
  **`submission_v2.csv` hanya valid kalau `LIMIT = None`.**
- Hasil per gambar di-cache: dijalankan ulang, yang sudah selesai tidak diulang. Cache dibuang
  otomatis kalau model atau pengaturan berubah.
- `debug_v2.csv`: jumlah kotak, conf, kotak, label orientasi, rotasi, teks per gambar. Crop yang
  sudah ditegakkan ada di `test_crops_v2/`.
- Baris tanpa hasil diisi `FALLBACK` karena Kaggle menolak sel kosong.

In [ ]:
import json

from ultralytics import YOLO

TEST_FOLDER = f"{DATA_FOLDER}/test"
SAMPLE_SUBMISSION = f"{DATA_FOLDER}/sample_submission.csv"
LIMIT = None       # UBAH: None = semua gambar, 200 = cek cepat
FALLBACK = "0"     # UBAH: Kaggle menolak sel kosong (null)
YOLO_WEIGHTS = f"{MODEL_FOLDER}/yolo_2cls.pt"
ORI_JSON = f"{MODEL_FOLDER}/orientation_v2.json"

if not os.path.exists(ORI_JSON):
    raise FileNotFoundError("orientation_v2.json belum ada - jalankan bagian 4 dulu.")
with open(ORI_JSON) as file:
    ori_saved = json.load(file)
ORI_MAP = {}
for label in ori_saved:
    ORI_MAP[int(label)] = ori_saved[label]
yolo = YOLO(YOLO_WEIGHTS)
get_paddle()
get_ori()
print("ENHANCE:", ENHANCE, "| ORI_MAP:", ORI_MAP)

In [ ]:
def empty_record(name):
    return {"id": name, "n_boxes": 0, "detected": 0, "conf": None, "box": None,
            "ori_label": None, "rot": None, "text": ""}


def process(name):
    record = empty_record(name)
    found = yolo.predict(source=f"{TEST_FOLDER}/{name}", imgsz=IMGSZ, conf=CONF, verbose=False)
    boxes = found[0].boxes
    best, record["n_boxes"] = best_entete(boxes.cls.tolist(), boxes.conf.tolist())
    if best is None:
        return record
    corners = boxes.xyxy[best].tolist()
    x1, y1, x2, y2 = int(corners[0]), int(corners[1]), int(corners[2]), int(corners[3])
    crop = found[0].orig_img[max(0, y1 - PAD): y2 + PAD, max(0, x1 - PAD): x2 + PAD]
    if crop.size == 0:
        return record
    label = ori_label(crop)
    upright = rotate_cw(crop, ORI_MAP.get(label, 0))
    cv2.imwrite(f"{TEST_CROPS_FOLDER}/{name}", upright)
    record.update(detected=1, conf=round(float(boxes.conf[best]), 3), box=f"{x1},{y1},{x2},{y2}",
                  ori_label=label, rot=ORI_MAP.get(label, 0), text=paddle_read(upright))
    return record


print(empty_record("contoh.jpg"))

In [ ]:
import pickle

CACHE = f"{WORK_FOLDER}/infer_cache_v2.pkl"   # dibuang otomatis kalau pengaturan/model berubah
signature = (ENHANCE, UPSCALE, CONF, PAD, os.path.getmtime(YOLO_WEIGHTS),
             os.path.getmtime(ORI_JSON))
records = {}
if os.path.exists(CACHE):
    with open(CACHE, "rb") as file:
        saved = pickle.load(file)
    if saved.get("sig") == signature:
        records = saved["items"]
print("dari cache:", len(records), "gambar")

sample = pd.read_csv(SAMPLE_SUBMISSION)
names = list(sample["id"])[:LIMIT]
for number, name in enumerate(tqdm(names)):
    if name not in records:
        records[name] = process(name)
    if number % 25 == 24:
        with open(CACHE, "wb") as file:
            pickle.dump({"sig": signature, "items": records}, file)
with open(CACHE, "wb") as file:
    pickle.dump({"sig": signature, "items": records}, file)

In [ ]:
debug_rows = []
for name in names:
    debug_rows.append(records[name])
debug = pd.DataFrame(debug_rows)
debug["text_empty"] = (debug["text"] == "").astype(int)
debug.to_csv(f"{WORK_FOLDER}/debug_v2.csv", index=False)

texts = []
n_empty = 0
for name in sample["id"]:
    text = ""
    if name in records:
        text = records[name]["text"]
    if text == "":
        n_empty = n_empty + 1
        text = FALLBACK
    texts.append(text)
submission = pd.DataFrame()
submission["id"] = sample["id"]
submission["text"] = texts
SUBMISSION_CSV = f"{WORK_FOLDER}/submission_v2.csv"
submission.to_csv(SUBMISSION_CSV, index=False)
print("terdeteksi:", int(debug["detected"].sum()), "/", len(debug),
      "| teks kosong (diisi fallback):", n_empty, "dari", len(submission))
rot_counts = debug.groupby("rot", dropna=False).size()
print(rot_counts.to_dict())

In [ ]:
check = pd.read_csv(SUBMISSION_CSV)   # baca ulang seperti yang dilakukan Kaggle
if check["text"].isna().any():
    raise ValueError("masih ada null")
if not submission["text"].str.fullmatch(r"[A-Z0-9/\-]+").all():
    raise ValueError("ada karakter terlarang")
print("submission OK:", SUBMISSION_CSV)
debug.head()